# UVERA · NB02 — AI-2 Scam Text Sentinel

| | |
|---|---|
| **Who runs it** | Teammate (P2) |
| **Accelerator** | GPU T4 ×2 (code uses 1 GPU) |
| **Internet** | On |
| **Secret** | `GEMINI_API_KEY` (Add-ons → Secrets). Fallback: a private dataset containing `gemini_key.txt` |
| **Inputs to add** | Dataset **uciml/sms-spam-collection-dataset** (Add Input → search “SMS Spam Collection Dataset”) |
| **Expected runtime** | ~45–90 min |
| **How to run** | **Save Version → Save & Run All (Commit)** — runs in the background, no 20-minute idle timeout, 12-hour cap |

**What this AI does (say this to judges):** A customer can paste a suspicious SMS or call script (Bangla, Banglish or English). AI-2 says if it looks like a scam, which kind, and how sure it is, and highlights the phrases that drove the decision. Training text is synthetic (templates + Gemini). The test uses a different writing style that was never seen in training, plus a public English SMS spam set as an outside check.

**Outputs** (in `/kaggle/working/outputs/NB02_outputs.zip`): `artifacts/ai2/` (tfidf_lr.joblib, emb_lr.npz, emb_evidential.pt, model_info.json, corpus.parquet), `reports/metrics_ai2.json`, figures

All data is synthetic or open-licensed. No real customer data.

In [ ]:
# 1) Get the UVERA code from the public GitHub repo (nothing to upload by hand)
REPO = "https://github.com/ABRUBAB/takaresolve-ai-hackathon-2026.git"
REF = "main"   # replace with a commit hash to reproduce an exact run
import os, sys, subprocess
if not os.path.exists("/tmp/uvera"):
    subprocess.run(["git", "clone", "-q", REPO, "/tmp/uvera"], check=True)
subprocess.run(["git", "-C", "/tmp/uvera", "fetch", "-q", "origin", REF], check=False)
subprocess.run(["git", "-C", "/tmp/uvera", "checkout", "-q", REF], check=True)
COMMIT = subprocess.run(["git", "-C", "/tmp/uvera", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
os.environ["UVERA_ROOT"] = "/tmp/uvera"
sys.path.insert(0, "/tmp/uvera/ml")   # use Kaggle's own numpy/pandas/sklearn/lightgbm (no reinstall)
print("UVERA code at commit", COMMIT)
print(f"To reproduce this exact run later, set REF = "{COMMIT}" in this cell.")

In [ ]:
# Kaggle already includes sentence-transformers and google-genai. Do NOT pip install them: upgrading them can
# replace PyTorch and break the GPU embedding model (this happened in an earlier run).
for mod in ("torch", "sentence_transformers", "google.genai"):
    try:
        m = __import__(mod, fromlist=["__version__"]); print(mod, getattr(m, "__version__", "ok"))
    except Exception as e:
        print(mod, "NOT AVAILABLE:", repr(e)[:200])

In [ ]:
# 2) Reproducibility: seeds, versions, time budget, output folder
import json
from IPython.display import Image, display
from uvera_ml.common import env_info, set_seed, Budget, out_dir, zip_outputs, write_json
set_seed(42)
budget = Budget(minutes=240)
OUT = out_dir("NB02")
info = env_info(); info["uvera_commit"] = COMMIT
write_json(OUT / "reports" / "versions_nb02.json", info)
print(json.dumps(info, indent=1))

In [ ]:
# Gemini key: Kaggle Secret -> env var -> gemini_key.txt in an attached PRIVATE dataset. The key is never printed.
from uvera_ml.brief.gemini import Gemini, load_key_on_kaggle
where = load_key_on_kaggle()
print("Gemini key:", where)
gem, gem_status = None, {"key_source": where}
if where != "not_found":
    try:
        gem = Gemini(cache_dir="/kaggle/working/gemini_cache")
        check = gem.ping()
        gem_status["check"] = check
        print("Gemini check:", check)
        if not check.get("ok"):
            gem = None
    except Exception as e:
        gem = None; gem_status["error"] = str(e)[:300]; print("Gemini not usable (quota/region/key):", str(e)[:200])
write_json(OUT / "reports" / "run_log_gemini.json", gem_status)  # diagnostics travel with the results zip

## Step A — build the text corpus (runs once; cached)
Seed templates (offline) + Gemini messages in two different style families (A = train, B = held-out test). If the Gemini key is missing, the notebook continues with templates only.

In [ ]:
import pandas as pd
from pathlib import Path
from uvera_ml.sim.text import template_corpus, gemini_corpus, load_uci_sms
corpus_path = Path("/kaggle/working/text_corpus.parquet")   # checkpoint: delete it to regenerate
if corpus_path.exists():
    corpus = pd.read_parquet(corpus_path)
else:
    parts = [template_corpus(per_template=40)]
    if gem is not None:
        parts.append(gemini_corpus(gem, n_per_call=60, calls_per_cell=2))  # 144 calls: fits the free daily quota
        print(gem.stats())
    else:
        print("Gemini not used: templates only (weaker text model). Fix the key and re-run if possible.")
    corpus = pd.concat(parts, ignore_index=True).drop_duplicates("text")
    corpus.to_parquet(corpus_path, index=False)
print(corpus.groupby(["source", "family"]).size())
display(corpus.sample(8, random_state=1)[["text", "label", "language", "family"]])
budget.check("corpus")

## Step B — train and compare three models
TF-IDF + logistic regression · BGE-M3 embeddings + logistic regression · BGE-M3 + evidential head (explicit “I don't know”). Validation: **StratifiedGroupKFold (5 folds, grouped by template / generation batch) × 3 seeds**. The served model is chosen on cross-validation, never on the test set.

In [ ]:
from uvera_ml.models.ai2 import run_ai2
try:
    uci = load_uci_sms()
except Exception as e:
    uci = None; print("UCI SMS set not available:", e)
summary = run_ai2(OUT, corpus, uci, use_bge=True, seeds=(42, 7, 1337),
                  diagnostics={"gemini": gem_status, "gemini_stats": gem.stats() if gem else None})
print("Diagnostics:", json.dumps(summary["diagnostics"], indent=1, default=str))
display(pd.DataFrame(summary["test_heldout_style_B"]))
print(json.dumps({"by_language": summary["test_by_language_served"], "external": summary["external_uci_sms_spam"]}, indent=1))
for ex in summary["examples"][:5]:
    print(round(ex["p_scam"], 2), ex["true"], "|", ex["text"][:90], "|", [h["phrase"] for h in ex["highlights"]])
display(Image(str(OUT / "reports" / "figures" / "ai2_reliability.png")))
budget.check("ai2")

In [ ]:
# Last step: package the results. Download the zip from the Output tab and send it to Rubab.
budget.check("done")
zip_outputs(OUT)